#  Proyecto Integrador: API de Ventas con Docker y CI/CD

Este notebook documenta, paso a paso, el proyecto integrador que combina
**Docker**, **contenerización de aplicaciones**, **principios de CI/CD** y
**automatización con GitHub Actions**.
**Integrantes:**
- Daniella Marissa Navarro Araniva
- Diego Alejando Escobar Barahona 

**Repositorio:**  
https://github.com/bapatata688/ci_cd_example.git
## Objetivo del proyecto

Construir una solución capaz de:

1. Ejecutar pruebas automáticamente.
2. Construir una imagen Docker.
3. Versionar la aplicación.
4. Publicar la imagen en un registro.
5. Preparar el despliegue automático.

## Escenario

Una empresa cuenta con una API en Python para consultar información de
ventas. Cada vez que un desarrollador actualiza el código se ejecutan
pruebas, se valida la sintaxis, se construye una imagen Docker, se publica
y se prepara para despliegue — todo de forma automática.

## Arquitectura general

```
GitHub → GitHub Actions → Pruebas → Docker Build → Docker Registry → Deploy
```


---
##  Estructura del Proyecto

```
ventas_api/
│
├── app.py                    # API Flask principal
├── requirements.txt          # Dependencias Python
│
├── tests/
│   └── test_app.py           # Pruebas automatizadas con pytest
│
├── Dockerfile                # Definición de la imagen Docker
│
└── .github/
    └── workflows/
        └── pipeline.yml      # Workflow de GitHub Actions (CI/CD)
```



---
##  Parte 1 — Desarrollo de la Aplicación

La aplicación es una **API REST con Flask** que expone un endpoint de salud (`/`) para confirmar que el servicio está operativo.

### ¿Por qué Flask?

- **Ligero**: mínima configuración, ideal para microservicios.
- **Pythonico**: sintaxis clara y sencilla.
- **Bien documentado**: gran ecosistema y comunidad.

### 1.1 Código de la API (`app.py`)

In [ ]:
# Instalación de dependencias en el entorno del notebook
%pip install flask pytest --quiet
print(" Dependencias instaladas correctamente.")

In [ ]:
from flask import Flask

# Inicializar la aplicación Flask
app = Flask(__name__)


@app.route("/")
def inicio():
    """Endpoint de salud: verifica que la API está operativa."""
    return {
        "mensaje": "API de ventas operativa",
        "status": "ok",
        "version": "1.0.0"
    }


print(" Aplicación Flask definida correctamente.")
print(f"   Rutas registradas: {[str(r) for r in app.url_map.iter_rules()]}")

### 1.2 Dependencias (`requirements.txt`)

```
flask
pytest
```

| Librería | Rol | Por qué |
|----------|-----|--------|
| `flask` | Framework web | Sirve la API HTTP |
| `pytest` | Framework de pruebas | Ejecuta y reporta las pruebas automatizadas |

### 1.3 Prueba automatizada (`tests/test_app.py`)

La prueba unitaria verifica que el endpoint devuelve **HTTP 200**, garantizando que la API responde correctamente.

```python
# tests/test_app.py
from app import app

def test_inicio():
    cliente = app.test_client()
    respuesta = cliente.get("/")
    assert respuesta.status_code == 200
```

La siguiente celda ejecuta la misma verificación directamente en el notebook:

In [ ]:
import json

def test_inicio():
    """Prueba que el endpoint raíz responde con HTTP 200."""
    cliente = app.test_client()
    respuesta = cliente.get("/")
    assert respuesta.status_code == 200, (
        f"Se esperaba 200, se obtuvo {respuesta.status_code}"
    )
    return respuesta.get_json()


try:
    resultado = test_inicio()
    print(" Prueba PASADA")
    print("   Respuesta JSON:", json.dumps(resultado, indent=2, ensure_ascii=False))
except AssertionError as e:
    print(f" Prueba FALLIDA: {e}")

---
##  Parte 2 — Contenerización con Docker

Docker empaqueta la aplicación y **todas sus dependencias** en una imagen portátil que se ejecuta de forma idéntica en cualquier entorno.

### ¿Por qué Docker?

| Problema sin Docker | Solución con Docker |
|---------------------|---------------------|
| "En mi máquina funciona" | El entorno está definido en el `Dockerfile` |
| Instalar Python manualmente | La imagen base ya lo incluye |
| Configuraciones inconsistentes | Imagen inmutable y reproducible |

### 2.1 Dockerfile

```dockerfile
# Imagen base con Python 3.12 preinstalado
FROM python:3.12

# Directorio de trabajo dentro del contenedor
WORKDIR /app

# Copiar dependencias primero (optimiza la caché de Docker)
COPY requirements.txt .

# Instalar dependencias
RUN pip install -r requirements.txt

# Copiar el resto del código fuente
COPY . .

# Exponer el puerto que usa Flask
EXPOSE 5000

# Comando de arranque
CMD ["python", "app.py"]
```

### 2.2 Capas del Dockerfile

```
┌─────────────────────────────────────┐
│  CMD ["python", "app.py"]           │  ← Comando de inicio
├─────────────────────────────────────┤
│  COPY . .                           │  ← Código fuente (cambia frecuente)
├─────────────────────────────────────┤
│  RUN pip install -r requirements    │  ← Dependencias (cambia poco)
├─────────────────────────────────────┤
│  COPY requirements.txt .            │  ← Lista de deps
├─────────────────────────────────────┤
│  WORKDIR /app                       │  ← Directorio de trabajo
├─────────────────────────────────────┤
│  FROM python:3.12                   │  ← Imagen base (reutilizada)
└─────────────────────────────────────┘
```

>  copiar `requirements.txt` antes que el código fuente aprovecha la caché de capas de Docker. Si solo cambia el código, Docker no reinstala las dependencias.

### 2.3 Comandos para prueba local

Ejecutar en la terminal (fuera del notebook):

```bash
# Construir la imagen
docker build -t ventas-api .

# Ejecutar el contenedor
docker run -p 5000:5000 ventas-api

# Verificar la respuesta
curl http://localhost:5000
```

Resultado esperado:

```json
{
  "mensaje": "API de ventas operativa"
}
```

---
## ⚙️ Parte 3 — Pipeline CI/CD con GitHub Actions

**CI/CD** (*Continuous Integration / Continuous Delivery*) automatiza las validaciones y el despliegue cada vez que el código cambia.

| Concepto | Significado | En este proyecto |
|----------|-------------|------------------|
| **CI** — Integración Continua | Verificar que el código nuevo no rompe nada | Job `test`: pruebas y validación de sintaxis |
| **CD** — Entrega Continua | Publicar automáticamente la versión aprobada | Job `build`: construir y subir imagen a Docker Hub |

### 3.1 Workflow completo (`.github/workflows/pipeline.yml`)

```yaml
name: CI/CD Pipeline

on:
  push:            # Se dispara con cualquier push al repositorio

jobs:

  # ── JOB 1: PRUEBAS ──────────────────────────────────────────────
  test:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4          # Descarga el código
      - uses: actions/setup-python@v5      # Instala Python 3.12
        with:
          python-version: "3.12"

      - name: Instalar Dependencias
        run: pip install -r requirements.txt

      - name: Ejecutar Pruebas                          # Falla aquí = pipeline detenido
        run: run: |
          PYTHONPATH=. pytest               

      - name: Validar Código                             # Detecta errores de sintaxis
        run: |
          python -m py_compile app.py
                     

  # ── JOB 2: BUILD Y PUBLICACIÓN ──────────────────────────────────
  build:
    needs: test                            # Solo corre si test pasó
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: docker/setup-buildx-action@v3
      - uses: docker/login-action@v3
        with:
          username: ${{ secrets.DOCKER_USERNAME }}
          password: ${{ secrets.DOCKER_PASSWORD }}
      - uses: docker/build-push-action@v6
        with:
          context: .
          push: true
          tags: daniellanav/ventas_api:latest
```

### 3.2 Descripción de cada etapa

#### Job `test`

| Paso | Acción | Si falla... |
|------|--------|-------------|
| `checkout@v4` | Descarga el repositorio en el runner | Pipeline se detiene |
| `setup-python@v5` | Instala Python 3.12 | Pipeline se detiene |
| Instalar Dependencias | `pip install -r requirements.txt` | Pipeline se detiene |
| Ejecutar Pruebas | `pytest` | **Pipeline se detiene** — la imagen no se publica |
| Validar Código | `python -m py_compile app.py` | Pipeline se detiene |

#### Job `build`

| Paso | Acción |
|------|--------|
| `checkout@v4` | Vuelve a descargar el repo (cada job es una VM nueva) |
| `setup-buildx-action@v3` | Activa BuildKit (construcción moderna y eficiente) |
| `login-action@v3` | Autenticación en Docker Hub con secretos encriptados |
| `build-push-action@v6` | Construye la imagen y la publica con `push: true` |

### 3.3 Configuración de secretos en GitHub

Las credenciales **nunca** se escriben en el código. Se guardan en:

```
GitHub → Settings → Secrets and variables → Actions → New repository secret
```

| Secret | Valor |
|--------|-------|
| `DOCKER_USERNAME` | Tu usuario de Docker Hub |
| `DOCKER_PASSWORD` | Tu contraseña o Access Token de Docker Hub |

>  **Por qué se uso Access Tokens:** Docker Hub permite crear tokens con permisos limitados (solo escritura). Si el token se compromete, se revoca sin cambiar la contraseña principal.

### 3.4 Flujo completo visualizado

```
git push
   │
   ▼
GitHub Actions se activa
   │
   ├─► JOB: test
   │     ├─ Checkout del código
   │     ├─ Configurar Python 3.12
   │     ├─ pip install -r requirements.txt
   │     ├─ pytest  ──── ¿Falla? → Pipeline detenido
   │     └─ py_compile  ─ ¿Falla? →  Pipeline detenido
   │
   └─► JOB: build  (solo si test pasó )
         ├─ Checkout del código
         ├─ Setup Docker Buildx
         ├─ Login en Docker Hub
         └─ Build + Push → daniellanav/ventas_api:latest 
```

---
##  Buenas Prácticas Aplicadas

| Práctica | Implementación en el proyecto |
|----------|-------------------------------|
|  **Secretos encriptados** | `DOCKER_USERNAME` y `DOCKER_PASSWORD` en GitHub Secrets, nunca en el código |
|  **Pruebas primero** | Ninguna imagen se publica si `pytest` falla |
|  **Dockerfile optimizado** | `requirements.txt` se copia antes que el código para aprovechar la caché |
|  **Jobs encadenados** | `build` depende de `test` con `needs:` — fallo temprano |
|  **Validación de sintaxis** | `py_compile` detecta errores antes de construir la imagen |

---
##  Mejoras Posibles

El pipeline actual es funcional. Estas son las mejoras recomendadas para un entorno de producción real:

| Mejora | Herramienta | Beneficio |
|--------|-------------|-----------|
| Cobertura de código | `pytest --cov` | Saber qué porcentaje del código está probado |
| Escaneo de vulnerabilidades | Bandit / Trivy | Detectar problemas de seguridad automáticamente |
| Calidad de código | SonarQube / Ruff | Mantener estándares de código consistentes |
| Versionamiento semántico | Tags `v1.0.0`, `v1.1.0` | Evitar depender solo de `latest`; permitir rollbacks |
| Despliegue automático | AWS ECS / Railway / Render | Ir del registro al servidor en producción |
| Notificaciones | Slack / Email | Alertar al equipo si el pipeline falla |

---
##  Aplicación en Ingeniería de Datos

El patrón **Checkout → Pruebas → Docker Build → Registro → Deploy** no es exclusivo de APIs web. En ingeniería de datos aplica directamente a:

```
INGENIERÍA DE DATOS          ESTE PROYECTO
──────────────────────       ──────────────────
Script ETL (Python)     ←→   app.py
Pruebas de datos        ←→   test_app.py
Imagen Docker del ETL   ←→   Dockerfile
Orquestador (Airflow)   ←→   GitHub Actions
Data Lake / DWH         ←→   Docker Hub
```

### Casos de uso concretos

| Caso | Descripción |
|------|-------------|
| **Procesos ETL** | Script Python que extrae, transforma y carga datos → contenerizado y desplegado automáticamente |
| **APIs analíticas** | FastAPI que sirve resultados de modelos ML → misma arquitectura CI/CD |
| **Procesamiento batch** | Jobs Pandas/Spark → Docker → pipeline CI/CD |
| **Reportes automáticos** | Scripts que generan dashboards o PDFs → despliegue programado |

---
##  Conclusiones

Este proyecto integrador demostró cómo combinar **Docker**, **Flask**, **pytest** y **GitHub Actions** en un único flujo automatizado.

### Logros alcanzados

1. **API funcional** con Flask, probada con pytest.
2. **Imagen Docker** reproducible y portátil.
3. **Pipeline CI/CD** que valida, construye y publica automáticamente.
4. **Seguridad básica** con secretos encriptados en GitHub.

### Lección clave

> La automatización no elimina el trabajo del desarrollador — lo enfoca en escribir código de calidad, mientras las validaciones y despliegues ocurren sin fricción.

Este mismo ciclo es el que usan organizaciones como Netflix, Spotify y cualquier empresa moderna con equipos de ingeniería de datos para garantizar entregas rápidas y confiables.